# 03 — Training and checkpoint audit

The supplied Referee checkpoint is the evaluation input. Training is optional and writes ignored files under `models/referee/checkpoints/`. This notebook inspects the checkpoint without loading prepared data or computing scores.


## Setup

```sh
uv sync --project models/referee --frozen
uv run --project models/referee/evaluation jupyter lab --ServerApp.root_dir=.
```


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import os
import subprocess

import pandas as pd
from IPython.display import display

from referee_evaluation import notebook_tools

REPOSITORY_ROOT = Path.cwd().resolve()
while not (REPOSITORY_ROOT / 'models' / 'referee' / 'recognition').is_dir():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
MODEL_PROJECT = REPOSITORY_ROOT / 'models' / 'referee'
print('Model project:', MODEL_PROJECT)


## Options

Training is disabled by default. The checkpoint inspection is read-only. Do not point `RUN_TRAINING` at the supplied checkpoint path when experimenting with a new run.


In [ ]:
RUN_TRAINING = False
INSPECT_CHECKPOINT = True
CHECKPOINT = MODEL_PROJECT / 'checkpoints' / 'referee-epoch=04-val_mean_ap=0.5564.ckpt'
PROFILE = 'sample'          # 'sample' or 'train'
PREPARED_DIR = MODEL_PROJECT / 'data' / '.local' / 'data' / PROFILE / 'prepared'
TRAIN_ARGS = ['--prepared-dir', str(PREPARED_DIR), '--max-epochs', '5']

print('Checkpoint:', CHECKPOINT)
print('Training enabled:', RUN_TRAINING)
print('Prepared data:', PREPARED_DIR)


## Readiness

The checkpoint file must exist; training additionally requires prepared manifests.


In [ ]:
print('Checkpoint present:', CHECKPOINT.is_file())
print('Evaluator present:', (MODEL_PROJECT / 'recognition' / 'checkpoint.py').is_file())
print('Prepared train manifest:', (PREPARED_DIR / 'manifests' / 'train.jsonl').is_file())
if RUN_TRAINING and not (PREPARED_DIR / 'manifests' / 'train.jsonl').is_file():
    raise FileNotFoundError('Prepare data with notebook 01 before training.')


## Inspect the checkpoint

This reports the 4-output head, epoch, step, Lightning version, hyperparameters and filename score. It does not load prepared data.


In [ ]:
MODEL_ENV = dict(os.environ)
MODEL_ENV.pop('PYTHONPATH', None)
MODEL_ENV.pop('PYTHONHOME', None)

if INSPECT_CHECKPOINT:
    result = subprocess.run(
        ['uv', 'run', '--project', str(MODEL_PROJECT), '--frozen', '--no-sync',
         'python', '-m', 'recognition.checkpoint', '--checkpoint', str(CHECKPOINT), '--json'],
        cwd=MODEL_PROJECT, env=MODEL_ENV, capture_output=True, text=True, check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    info = json.loads(result.stdout)
    display(pd.DataFrame([info]).T)
    print('Head classes:', info['num_classes'], 'in features:', info['head_in_features'])
else:
    print('Checkpoint inspection disabled.')


## Optional training

This cell is disabled by default and writes new ignored checkpoints. Use a separate checkpoint root if you do not want to mix with the supplied evaluation input.


In [ ]:
if RUN_TRAINING:
    command = ['uv', 'run', '--project', str(MODEL_PROJECT), '--frozen',
               'python', '-m', 'training.train', *TRAIN_ARGS]
    print('Launching:', ' '.join(command))
    if not notebook_tools.run_interruptible_command(command, cwd=MODEL_PROJECT, env=MODEL_ENV):
        raise RuntimeError('Training interrupted.')
else:
    print('Training disabled.')


## Next

Open `04_validation_evaluation.ipynb` after the checkpoint is confirmed.
